# Paquete para el equipo: revisión de cajas (RE3.3), sesión cronometrada (RE3.2) y calidad del corpus

Arma lo que se envía al equipo del 8 Primates Project y lo que se queda aquí para leer lo que
devuelva. Todo sale de la partición de prueba, que el modelo no vio: no se pide ningún dato
nuevo, sólo el juicio y el tiempo de una persona. Las cajas son las que entrega la herramienta
(`detect.py`) con el modelo seleccionado (`yolo26s_v3`) en su umbral de validación: una tabla
de Raven por grabación, con los duplicados entre ventanas ya fundidos.

Las reglas (umbral, emparejamiento, estratos, tamaños, orden de la sesión) son las de
`../protocolo_oe3.md`: si cambian aquí, cambian allá, y **antes** de enviar el paquete.

Escribe en `../revision/`:

| Archivo | Qué es | Se envía |
|---|---|---|
| `detecciones_prueba.csv` | las tablas de la herramienta, todas juntas (caché: borrarla las rehace) | no |
| `poblacion.csv` | cada detección con su categoría de RE3.3 frente a las anotaciones | no |
| `clave_muestra.csv` | qué es cada caja de la muestra: estrato, categoría y score | **no: la revisión es ciega** |
| `clave_calidad.csv` | qué es cada caja de la tarea de calidad: tipo de hallazgo de CLOD | **no** |
| `sesion_bloques.csv` | las grabaciones de cada carpeta de la sesión, con su densidad | no |
| `paquete_equipo/` | lo que se envía, con una página de instrucciones en inglés | sí |

In [ ]:
import json
import shutil
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf

PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_DIR / "src"))

from analysis.review import NONE, categorize, load_annotations, relative, tool_tables, touches
from core.config import RAW_DIR, SEED
from data.annotations import species_of
from data.raven import BEGIN, CALL, END, HIGH, LOW, SCORE, SPECIES, renumber
from data.species import split_label
from evaluation.evaluator import RawPredictions
from evaluation.metrics import window_classes
from models.registry import load_checkpoint

RUN = "yolo26s_v3"  # el modelo seleccionado en el capítulo 5
RUN_DIR = PROJECT_DIR / "runs" / RUN
OUT = Path.cwd().parent / "revision"
PACKAGE = OUT / "paquete_equipo"
comparison = json.loads((PROJECT_DIR / "runs/comparacion/comparacion_modelos_v3.json").read_text())
THRESHOLD = next(m for m in comparison["models"] if m["model"] == RUN)["threshold"]

# La muestra, en cajas por estrato. «Nada anotado» va por tramos de score y pesa más el de alta
# confianza, el que pide el IOV de RE3.3. Los aciertos son controles: van mezclados sin aviso y
# miden qué tan exigente es el revisor.
SCORES = {"baja": (THRESHOLD, 0.3), "media": (0.3, 0.5), "alta": (0.5, 1.01)}
SAMPLE = {
    "none_alta": 300,
    "none_media": 200,
    "none_baja": 133,
    "box": 220,  # todas: son pocas
    "label": 87,  # todas
    "match": 60,
}
# Los hallazgos de CLOD de la tarea de calidad: en validación, del modelo final; en train, del
# k-fold, que está en el servidor (`python src/find_issues.py runs/yolo26s_v3_kfold5/...`)
FINDINGS = {
    "val": RUN_DIR / "hallazgos_val.csv",
    "train": PROJECT_DIR / "runs/yolo26s_v3_kfold5/hallazgos_train.csv",
}
# La sesión: cuatro carpetas parecidas, con una grabación de cada especie, que una persona hace
# en este orden. Manual, asistida, asistida, manual: el aprendizaje a lo largo de la sesión cae
# parejo en las dos condiciones.
ORDER = ["manual", "assisted", "assisted", "manual"]
PER_SPECIES, MAX_DURATION_S = 2, 120  # grabaciones de cada especie por carpeta
rng = np.random.default_rng(SEED)

## La población: lo que la herramienta escribe en prueba, frente a lo anotado

Cada detección se clasifica con la regla de RE3.3 (`analysis.review.categorize`), en tiempo de
grabación: acierto si toma una anotación de su clase con IoU ≥ 0,3; si no, caja distinta, otra
llamada o nada anotado. La referencia son las anotaciones de las clases del modelo que no
quedaron en revisión, como en el capítulo 5. Las clases de ventana (`as/hc`, `pt/dc`) quedan
fuera, como en RE3.3. Una detección «nada anotado» que cae sobre una fila en revisión, una frase
o un tipo de llamada sin clase no se revisa: ahí ya hay algo anotado.

In [ ]:
dump = RawPredictions.load(RUN_DIR / f"{RUN}_test_predictions.pt")
names = dump.labels
recordings = sorted(relative(name) for name in dump.recording_names)
dataset = json.loads((RUN_DIR / "config.json").read_text())["dataset"]
annotations = load_annotations(dataset["joined_labels"])
annotations = annotations[annotations.recording.isin(recordings)]
reference = annotations[~annotations.requires_review & annotations.label.isin(names)]
window_labels = {names[c] for c in window_classes(dump.truth, len(names))}

cache = OUT / "detecciones_prueba.csv"
if cache.exists():
    detections = pd.read_csv(cache)
else:
    loaded = load_checkpoint(RUN_DIR / "best.pt")
    detections = tool_tables(loaded, recordings, THRESHOLD)
    detections.to_csv(cache, index=False)

population = categorize(detections, reference, names)
population = population[~population.label.isin(window_labels)].copy()
population["species"] = [split_label(label)[0] for label in population.label]
population["other_annotation"] = touches(population, annotations.drop(reference.index))
tier = pd.cut(population[SCORE], [low for low, _ in SCORES.values()] + [1.01], labels=list(SCORES))
population["stratum"] = population.category.where(
    population.category != NONE, NONE + "_" + tier.astype(str)
)
population.loc[(population.category == NONE) & population.other_annotation, "stratum"] = "other"
population.to_csv(OUT / "poblacion.csv", index=False)

hours = sum(sf.info(str(RAW_DIR / r)).duration for r in recordings) / 3600
test_info = {"run": RUN, "threshold": THRESHOLD, "recordings": len(recordings), "hours": hours}
(OUT / "resumen_prueba.json").write_text(json.dumps(test_info, indent=1))
summary = population.stratum.value_counts().rename("detecciones").to_frame()
summary["por hora"] = (summary.detecciones / hours).round(1)
summary["a revisar"] = pd.Series(SAMPLE)
print(f"{len(recordings)} grabaciones, {hours:.2f} h, umbral {THRESHOLD}")
summary

## La muestra de la revisión de cajas

Muestreo aleatorio dentro de cada estrato, con semilla fija. El revisor no ve el score ni la
categoría: sólo la caja y la especie y llamada propuestas, igual para todas. Las grabaciones van
en orden aleatorio y, dentro de cada una, las cajas por tiempo, para abrir cada audio una vez.

In [ ]:
strata = population.groupby("stratum")
picked = pd.concat(
    strata.get_group(s).sample(n=min(n, len(strata.get_group(s))), random_state=SEED)
    for s, n in SAMPLE.items()
)
shuffled = {r: i for i, r in enumerate(rng.permutation(picked.recording.unique()))}
picked = picked.sort_values(BEGIN).sort_values(
    "recording", key=lambda r: r.map(shuffled), kind="stable"
)
picked["ID"] = [f"C{i:03d}" for i in range(1, len(picked) + 1)]
picked["file"] = [Path(r).name for r in picked.recording]
assert picked.groupby("file").recording.nunique().max() == 1, "dos grabaciones con el mismo nombre"

# La clave: lo que el revisor no ve, más la anotación a que se refiere cada caja
picked["annotated_as"] = picked.annotation.map(reference.label)
key = [
    "ID",
    "stratum",
    "category",
    SCORE,
    "label",
    "annotated_as",
    "recording",
    BEGIN,
    END,
    LOW,
    HIGH,
]
picked[key].rename(columns={SCORE: "score"}).to_csv(OUT / "clave_muestra.csv", index=False)

check = PACKAGE / "1_check_boxes"
shutil.rmtree(check, ignore_errors=True)
(check / "audio").mkdir(parents=True)
sheet = pd.DataFrame(
    {
        "ID": picked.ID,
        "file": picked.file,
        "start_s": picked[BEGIN].round(1),
        "model_says": picked[SPECIES] + " " + picked[CALL],
        "answer": "",
        "comment": "",
    }
)
sheet.to_csv(check / "boxes.csv", index=False)

# Cada audio, con una tabla de Raven al lado con sólo sus cajas y su ID
for recording, items in picked.groupby("recording"):
    shutil.copy2(RAW_DIR / str(recording), check / "audio" / Path(str(recording)).name)
    table = renumber(items[["View", "Channel", BEGIN, END, LOW, HIGH, "ID", SPECIES, CALL]].copy())
    path = check / "audio" / f"{Path(str(recording)).stem}.boxes.txt"
    table[["Selection", *table.columns[:-1]]].to_csv(path, sep="\t", index=False)

print(f"{len(picked)} cajas en {picked.recording.nunique()} grabaciones")
pd.crosstab(picked.stratum, picked.species)

## La calidad del corpus: los hallazgos de CLOD, para que el equipo los resuelva

CLOD (`src/find_issues.py`) ya marcó, sobre predicciones fuera de muestra, las anotaciones que
parecen faltar (`missing`: el modelo ve con confianza una llamada sin anotar), sobrar
(`spurious`: ninguna predicción la toca), tener otra etiqueta (`label`) o estar mal ubicadas
(`location`). En train las predicciones son del k-fold (cada grabación, del modelo que no la
vio); en validación, del modelo final. Prueba queda fuera: ya está en la revisión de cajas.

Cada hallazgo es una caja: la del modelo en `missing`, la anotación original en los otros tres.
El revisor la juzga con las mismas cinco palabras de la revisión de cajas, sin saber si es del
modelo o suya. Van todos, intercalados por tipo y al azar dentro de cada tipo: lo que el equipo
alcance a revisar, desde el principio, es una muestra al azar de cada tipo.

In [ ]:
for split, path in FINDINGS.items():
    if not path.exists():
        print(f"falta {split}: {path.relative_to(PROJECT_DIR)} (copiarla del servidor y repetir)")
findings = pd.concat(
    pd.read_csv(path).assign(split=split) for split, path in FINDINGS.items() if path.exists()
)
findings["recording"] = findings.recording.map(relative)
findings = findings[~findings.recording.isin(recordings)]
findings = findings.sample(frac=1, random_state=SEED)
findings["turn"] = findings.groupby("Hallazgo").cumcount()
findings = findings.sort_values(["turn", "Hallazgo"], kind="stable").reset_index(drop=True)
findings["ID"] = [f"Q{i:05d}" for i in range(1, len(findings) + 1)]
findings["file"] = [Path(r).name for r in findings.recording]
findings["folder"] = [Path(r).parent.as_posix() for r in findings.recording]
findings.drop(columns=["turn"]).to_csv(OUT / "clave_calidad.csv", index=False)

quality = PACKAGE / "3_dataset_quality"
shutil.rmtree(quality, ignore_errors=True)
(quality / "tables").mkdir(parents=True)
call = [split_label(label) for label in findings.species]
pd.DataFrame(
    {
        "ID": findings.ID,
        "folder": findings.folder,
        "file": findings.file,
        "start_s": findings.begin_time_s.round(1),
        "box_says": [f"{sp.upper()} {ct.upper()}" for sp, ct in call],
        "answer": "",
        "comment": "",
    }
).to_csv(quality / "boxes.csv", index=False)

raven = findings.rename(
    columns=dict(
        zip(
            ["begin_time_s", "end_time_s", "low_freq_hz", "high_freq_hz"],
            [BEGIN, END, LOW, HIGH],
            strict=True,
        )
    )
).assign(
    View="Spectrogram 1",
    Channel=1,
    **{SPECIES: [sp.upper() for sp, _ in call], CALL: [ct.upper() for _, ct in call]},
)
for recording, items in raven.groupby("recording"):
    table = renumber(items[["View", "Channel", BEGIN, END, LOW, HIGH, "ID", SPECIES, CALL]].copy())
    path = (
        quality
        / "tables"
        / Path(str(recording)).parent.name
        / f"{Path(str(recording)).stem}.quality.txt"
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    table[["Selection", *table.columns[:-1]]].to_csv(path, sep="\t", index=False)
print(f"{len(findings)} hallazgos en {findings.recording.nunique()} grabaciones")
pd.crosstab(findings.Hallazgo, findings.split)

## Las carpetas de la sesión cronometrada

Fuera de la sesión quedan las grabaciones de la muestra (quien las revisó las conoce), las de
más de dos minutos, las sin anotaciones y las de especies sin ningún tipo entrenado (*Cebus*:
ahí la condición asistida no propondría nada). Cada carpeta lleva `PER_SPECIES` grabaciones de
cada especie, así las cuatro tienen la misma mezcla; de muchas elecciones al azar se queda la que las
deja más parecidas en minutos y en anotaciones: con un solo revisor, una carpeta más densa en
una condición la perjudicaría.

In [ ]:
trained_species = {split_label(name)[0] for name in names}
per_recording = pd.DataFrame({"recording": recordings}).assign(
    species=lambda d: [species_of(RAW_DIR / r) for r in d.recording],
    duration_s=lambda d: [sf.info(str(RAW_DIR / r)).duration for r in d.recording],
    n_annotations=lambda d: d.recording.map(annotations.recording.value_counts()).fillna(0),
    n_proposals=lambda d: d.recording.map(detections.recording.value_counts()).fillna(0),
)
per_recording["density"] = per_recording.n_annotations / per_recording.duration_s * 60
candidates = per_recording[
    ~per_recording.recording.isin(picked.recording)
    & (per_recording.duration_s <= MAX_DURATION_S)
    & (per_recording.n_annotations > 0)
    & per_recording.species.isin(trained_species)
]
n_folders = len(ORDER)
groups = [g for _, g in candidates.groupby("species") if len(g) >= PER_SPECIES * n_folders]
steps = np.tile(np.arange(n_folders) + 1, PER_SPECIES)

# De muchas elecciones al azar (`PER_SPECIES` grabaciones de cada especie en cada carpeta), la que
# deja las cuatro carpetas más parecidas en minutos de audio y en anotaciones
best, best_cost = [], np.inf
for _ in range(20_000):
    chosen = [
        g.iloc[rng.choice(len(g), len(steps), replace=False)].assign(step=steps) for g in groups
    ]
    picks = pd.concat(chosen)
    totals = [
        picks.groupby("step")[column].sum().to_numpy() for column in ("duration_s", "n_annotations")
    ]
    cost = sum(np.ptp(t) / t.mean() for t in totals)
    if cost < best_cost:
        best, best_cost = chosen, cost
blocks = pd.concat(best)
blocks["condition"] = [ORDER[s - 1] for s in blocks.step]
blocks["folder"] = [f"{s}_{c}" for s, c in zip(blocks.step, blocks.condition, strict=True)]
blocks = blocks.sort_values(["step", "species", "recording"])
blocks.to_csv(OUT / "sesion_bloques.csv", index=False)
folders = blocks.groupby("folder").agg(
    recordings=("recording", "size"),
    minutes=("duration_s", lambda s: s.sum() / 60),
    annotations=("n_annotations", "sum"),
    proposals=("n_proposals", "sum"),
)
folders["annotations_per_min"] = folders.annotations / folders.minutes
folders.round(1)

In [ ]:
session = PACKAGE / "2_timed_session"
shutil.rmtree(session, ignore_errors=True)
for recording, folder, condition in zip(
    blocks.recording, blocks.folder, blocks.condition, strict=True
):
    (session / folder).mkdir(parents=True, exist_ok=True)
    shutil.copy2(RAW_DIR / recording, session / folder / Path(recording).name)
    if condition == "assisted":  # la tabla de la herramienta tal cual, con su score
        table = detections[detections.recording == recording].drop(columns=["recording", "label"])
        path = session / folder / f"{Path(recording).stem}.model.txt"
        renumber(table.copy()).to_csv(path, sep="\t", index=False)
pd.DataFrame({"folder": blocks.folder.unique(), "start_time": "", "end_time": ""}).assign(
    minutes_paused="", notes=""
).to_csv(session / "times.csv", index=False)

## Las instrucciones

Una página, de `plantilla_instrucciones.md` con las cifras de esta corrida.

In [ ]:
template = (OUT / "plantilla_instrucciones.md").read_text()
folder_minutes = blocks.groupby("folder").duration_s.sum() / 60
(PACKAGE / "INSTRUCTIONS.md").write_text(
    template.format(
        n_boxes=len(picked),
        check_hours=f"{len(picked) // 60}–{len(picked) // 60 + 2}",
        n_findings=f"{len(findings):,}",
        n_folders=n_folders,
        folder_minutes=f"{folder_minutes.mean():.0f}",
        threshold=f"{THRESHOLD:.2f}",
    )
)
size = sum(f.stat().st_size for f in PACKAGE.rglob("*") if f.is_file()) / 1e6
print(f"paquete: {size:.0f} MB en {PACKAGE.relative_to(PROJECT_DIR)}")